# Experiment: 2D cond 1D — Infeasible / Adversarial Targets

**Reviewer question:** the main experiments (`Exp_2D_cond_1D.ipynb`) build the target `G(Y)` by literally slicing the true joint `P(X,Y)` at a chosen `x_star`, so a perfect match (`L(x_star) = 0`) is guaranteed by construction. This notebook asks: **what happens when no such `x*` exists** — when `G` cannot be exactly realized by any `x`?

This notebook reuses the same pretrained checkpoints, joint GMM, and optimization code (`Optimization.optimize_LGD`) as `Exp_2D_cond_1D.ipynb` — nothing in `src/` is modified. The only change is *what target `G` we hand to the optimizer*, because `optimize_LGD` already treats `(mog_means, mog_variances, weights)` as a free-standing target fully decoupled from the joint.

### The key trick: a cheap analytic oracle
Because `dim(x) = 1` here, the reachable set `{P(Y|X=x) : x in R}` is a 1-D curve through distribution space, and every point on it is closed-form Gaussian (`compute_conditionals` + `compute_alpha`). So for **any** target `G` — feasible or not — we can grid-search `x` directly on the analytic conditional and compute
```
L*(G) = min_x  gmm_l2_distance( P(Y|X=x), G )
```
in milliseconds, with no diffusion model involved. `L*` is the ground truth we grade MLGD / MLGD-F against below. On a genuinely feasible target `L* = 0`; we use this as a sanity check.

### Scenarios
| id | name | construction | intent |
|---|---|---|---|
| S0 | feasible baseline | true conditional at `x_star=-5` (same as main experiment) | sanity check, `L*` should be ≈0 |
| S1 | soft / near-feasible | true conditional at `x_star=-5`, component means shifted by `+SHIFT_S1` | mild, controllable infeasibility |
| S2 | structural / bimodal | `0.5 * P(Y\|X=x1) + 0.5 * P(Y\|X=x2)` for well-separated `x1, x2` (mirrors the protein-folding motivating example in the paper's own introduction) | target requires two conditionals at once |
| S3 | hard / out-of-support | Gaussian centered far outside the range `Y` ever takes under the joint | tests graceful degradation vs. pathological behavior at the extreme |


In [ ]:
import os
# ============================================================
# CONFIG
# Reuses the SAME joint GMM + checkpoints as EXPERIMENT_NAME below (feasible-target notebook).
# Results from THIS notebook are written to a separate results dir so nothing is overwritten.
# ============================================================
BASE_EXPERIMENT_NAME = "2D_cond_1D"          # source of GMM params + model checkpoints (unchanged)
EXPERIMENT_NAME       = "2D_cond_1D_infeasible"   # where THIS notebook writes its own results
GLOBAL_SEED           = 42

BASE_DIR        = os.path.normpath(os.path.join(os.getcwd(), ".."))
PARAMS_DIR      = f"{BASE_DIR}/params"
CHECKPOINT_DIR  = f"{BASE_DIR}/checkpoints/{BASE_EXPERIMENT_NAME}"   # reuse existing checkpoints, no retraining
RESULTS_DIR     = f"{BASE_DIR}/results/{EXPERIMENT_NAME}"

# Architecture (must match the checkpoints being loaded)
NBLOCKS, NUNITS       = 3, 128
NBLOCKS_CM, NUNITS_CM = 3, 128
DIFFUSION_STEPS       = 100
CONDITION_ON          = 1   # dim(x)=1, dim(y)=1

# Optimization (kept modest: 3 scenarios x 2 methods x N_ATTEMP_OPTIM runs)
N_ATTEMP_OPTIM             = 10
NSAMPLES_IN_OPTIM_FOR_MMD  = 250
NUM_X_T_LGD                = 3
NUM_X_T_LGD_CM              = 3

# Scenario knobs
S1_MEAN_SHIFT = 1.0     # S1: shift true-conditional component means by this much
S2_X1, S2_X2  = -5.0, 5.0   # S2: the two x-locations mixed together
S2_WEIGHT     = 0.5         # S2: mixture weight on the x1 conditional
S3_MEAN       = 20.0    # S3: target mean, well outside the joint's Y-range (~[-8, 8])
S3_VAR        = 0.3     # S3: target variance (typical of a single component)

# Oracle grid search
ORACLE_GRID_LO, ORACLE_GRID_HI, ORACLE_GRID_N = -15.0, 30.0, 900
ORACLE_REFINE_N = 400


In [ ]:
import os, sys
src_path = os.path.normpath(os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "src"))
if src_path not in sys.path:
    sys.path.insert(0, src_path)
print(f"src path on sys.path: {src_path}")


In [ ]:
# Install dependencies if needed
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "flow_matching", "POT", "-q"])


In [ ]:
import os, sys, time, json
import importlib
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
from functools import partial
from tqdm import trange

import Diffusion
import LossFunctions
import ConsistencyModels
import dist_utils
import Optimization
import experiment_utils
from ConsistencyModels import ConsistencyModeliCT
from LossFunctions import MMDLoss, RBF

for mod in [Diffusion, LossFunctions, ConsistencyModels,
            dist_utils, Optimization, experiment_utils]:
    importlib.reload(mod)

os.makedirs(RESULTS_DIR, exist_ok=True)
print("Imports done.")


In [ ]:
env_info = experiment_utils.get_environment_info()
experiment_utils.print_environment_info(env_info)


In [ ]:
experiment_utils.set_global_seed(GLOBAL_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## Load the joint GMM (same one `Exp_2D_cond_1D.ipynb` uses)

In [ ]:
loaded = experiment_utils.load_gmm_params(PARAMS_DIR, BASE_EXPERIMENT_NAME)
if loaded is None:
    raise FileNotFoundError(
        f"Could not find saved GMM params for '{BASE_EXPERIMENT_NAME}' in {PARAMS_DIR}. "
        "Run Exp_2D_cond_1D.ipynb at least once first (it generates and saves the joint GMM)."
    )
mu_list, Sigma_list, alpha, mog_means_true, mog_variances_true, weights_true, x_star = loaded
mu_list    = [mu.float() for mu in mu_list]
Sigma_list = [cov.float() for cov in Sigma_list]
alpha      = alpha.float()
print(f"Loaded joint GMM with {len(mu_list)} components. Reference x_star = {x_star.item()}")


## Load pretrained models (no retraining — reuses `2D_cond_1D` checkpoints)

In [ ]:
B_dummy = torch.cat([m.view(1, -1) for m in mu_list], dim=0)
nfeatures = B_dummy.shape[1] - CONDITION_ON

Cos_ConsistencyModeliCT = ConsistencyModeliCT(
    nfeatures=nfeatures, condition_on=CONDITION_ON, nunits=NUNITS_CM, depth=NBLOCKS_CM
)
ok_cm = experiment_utils.load_checkpoint_with_hf_fallback(
    Cos_ConsistencyModeliCT, "CM", CHECKPOINT_DIR, BASE_EXPERIMENT_NAME, GLOBAL_SEED, device
)
assert ok_cm, "Failed to load the CM (MLGD-F) checkpoint — run Exp_2D_cond_1D.ipynb once first."

model_cond = Diffusion.DiffusionModel(
    nfeatures=B_dummy.shape[1], nblocks=NBLOCKS, nunits=NUNITS,
    condition=True, condition_on=CONDITION_ON, diffusion_steps=DIFFUSION_STEPS
)
ok_cond = experiment_utils.load_checkpoint_with_hf_fallback(
    model_cond, "Diffusion_cond", CHECKPOINT_DIR, BASE_EXPERIMENT_NAME, GLOBAL_SEED, device
)
assert ok_cond, "Failed to load the conditional diffusion (MLGD) checkpoint."

model_uncond = Diffusion.DiffusionModel(
    nfeatures=CONDITION_ON, nblocks=NBLOCKS, nunits=NUNITS,
    condition=False, diffusion_steps=DIFFUSION_STEPS
)
ok_uncond = experiment_utils.load_checkpoint_with_hf_fallback(
    model_uncond, "Diffusion_uncond", CHECKPOINT_DIR, BASE_EXPERIMENT_NAME, GLOBAL_SEED, device
)
assert ok_uncond, "Failed to load the unconditional diffusion checkpoint."
print("All checkpoints loaded.")


## Build the infeasible / adversarial targets (S1, S2, S3)

Each target is just a `(means, variances, weights)` GMM over `Y` — exactly the same object type `Exp_2D_cond_1D.ipynb` builds from a single `x_star`. `optimize_LGD` doesn't know or care where it came from.

In [ ]:
def true_conditional_at(x_val, threshold=0.01):
    """Analytic P(Y|X=x_val) for the joint GMM, filtered/normalized (same as main notebook)."""
    x = torch.tensor([float(x_val)])
    mu_c, S_c = dist_utils.compute_conditionals(mu_list, Sigma_list, x)
    w_c = dist_utils.compute_alpha(mu_list, Sigma_list, alpha, x)
    return dist_utils.filter_and_normalize(mu_c, S_c, w_c, threshold=threshold)

def scenario_loss(x_val, tgt_mu, tgt_S, tgt_w):
    """Exact L2-GMM loss L(x) = || P(Y|X=x) - G ||^2 at a scalar x, closed form (no sampling)."""
    mu_c, S_c, w_c = true_conditional_at(x_val)
    return dist_utils.gmm_l2_distance(mu_c, S_c, w_c, tgt_mu, tgt_S, tgt_w)

def oracle_grid_search(tgt_mu, tgt_S, tgt_w,
                        lo=ORACLE_GRID_LO, hi=ORACLE_GRID_HI,
                        n=ORACLE_GRID_N, n_refine=ORACLE_REFINE_N):
    """L*(G) = min_x L(x), found by dense grid search then a local refine pass."""
    xs = torch.linspace(lo, hi, n)
    losses = torch.tensor([scenario_loss(x.item(), tgt_mu, tgt_S, tgt_w) for x in xs])
    i = int(losses.argmin())
    lo2 = xs[max(i - 2, 0)].item()
    hi2 = xs[min(i + 2, n - 1)].item()
    xs2 = torch.linspace(lo2, hi2, n_refine)
    losses2 = torch.tensor([scenario_loss(x.item(), tgt_mu, tgt_S, tgt_w) for x in xs2])
    j = int(losses2.argmin())
    return xs2[j].item(), losses2[j].item(), (xs, losses)   # also return the full curve for plotting


In [ ]:
scenarios = {}

# --- S0: feasible baseline (sanity check — L* should be ~0 at x_star) ---
mu0, S0_, w0 = true_conditional_at(x_star.item())
scenarios["S0_feasible_baseline"] = dict(mog_means=mu0, mog_variances=S0_, weights=w0)

# --- S1: soft / near-feasible — true conditional at x_star, component means shifted ---
mu1, S1_, w1 = true_conditional_at(x_star.item())
mu1 = mu1 + S1_MEAN_SHIFT
scenarios["S1_soft_shift"] = dict(mog_means=mu1, mog_variances=S1_, weights=w1)

# --- S2: structural / bimodal — 0.5/0.5 mixture of two well-separated conditionals ---
mu_a, Sa, wa = true_conditional_at(S2_X1)
mu_b, Sb, wb = true_conditional_at(S2_X2)
mu2 = torch.cat([mu_a, mu_b], dim=0)
S2_ = torch.cat([Sa, Sb], dim=0)
w2  = torch.cat([S2_WEIGHT * wa / wa.sum(), (1 - S2_WEIGHT) * wb / wb.sum()], dim=0)
scenarios["S2_bimodal_mixture"] = dict(mog_means=mu2, mog_variances=S2_, weights=w2)

# --- S3: hard / out-of-support — Gaussian far outside the joint's Y-range ---
mu3 = torch.tensor([[S3_MEAN]])
S3_ = torch.tensor([[[S3_VAR]]])
w3  = torch.tensor([1.0])
scenarios["S3_hard_out_of_support"] = dict(mog_means=mu3, mog_variances=S3_, weights=w3)

# --- Oracle for every scenario ---
for name, sc in scenarios.items():
    x_oracle, L_star, curve = oracle_grid_search(sc["mog_means"], sc["mog_variances"], sc["weights"])
    sc["x_oracle"], sc["L_star"], sc["curve"] = x_oracle, L_star, curve
    print(f"{name:26s} | x_oracle={x_oracle:7.3f} | L*={L_star:.6f}")


## Loss curve `L(x)` per scenario

Since `L(x)` is closed-form, we can plot the *entire* loss landscape over `x` for each target — this is the cheapest possible diagnostic for whether MLGD / MLGD-F land near the true global minimum or get stuck somewhere else.

In [ ]:
fig, axes = plt.subplots(1, len(scenarios), figsize=(5 * len(scenarios), 4), sharey=False)
for ax, (name, sc) in zip(axes, scenarios.items()):
    xs, losses = sc["curve"]
    ax.plot(xs.numpy(), losses.numpy(), lw=1.5)
    ax.axvline(sc["x_oracle"], color="red", ls="--", label=f"x_oracle={sc['x_oracle']:.2f}")
    ax.set_title(f"{name}\nL*={sc['L_star']:.4f}")
    ax.set_xlabel("x"); ax.set_ylabel("L(x)"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


## Optimize under each target: MLGD vs MLGD-F

Identical call to `Optimization.optimize_LGD` as the main notebook — only the `(mog_means, mog_variances, weights)` argument changes per scenario.

In [ ]:
def run_method(model_cond_obj, CM_flag, num_x_t, sc, n_attempts, seed_offset=0):
    x_preds, l2_gmm_list, l2_xstar_list, l2_xoracle_list, times, final_losses = [], [], [], [], [], []
    for i in trange(n_attempts, leave=False):
        run_seed = experiment_utils.set_run_seed(GLOBAL_SEED + seed_offset, i)
        t0 = time.time()
        best_x_t, _, final_loss = Optimization.optimize_LGD(
            model_uncond, model_cond_obj,
            sc["mog_means"], sc["mog_variances"], sc["weights"],
            mu_list, Sigma_list, alpha,
            nsamples=NSAMPLES_IN_OPTIM_FOR_MMD, loss="MMD", device=device,
            CM=CM_flag, FLAG=False, num_x_t=num_x_t,
        )
        times.append(time.time() - t0)
        final_losses.append(final_loss)
        x_preds.append(best_x_t)

        x_pred = best_x_t.float().view(-1).cpu()
        mu_p, S_p, w_p = true_conditional_at(x_pred.item())
        l2_gmm = dist_utils.gmm_l2_distance(mu_p, S_p, w_p, sc["mog_means"], sc["mog_variances"], sc["weights"])

        l2_gmm_list.append(l2_gmm)
        l2_xstar_list.append((x_pred - x_star.float().cpu()).pow(2).sum().sqrt().item())
        l2_xoracle_list.append(abs(x_pred.item() - sc["x_oracle"]))
    return dict(x_pred=x_preds, l2_gmm=l2_gmm_list, l2_xstar=l2_xstar_list,
                l2_xoracle=l2_xoracle_list, times=times, final_loss=final_losses)

all_results = {}
for name, sc in scenarios.items():
    print(f"\n=== {name} (L*={sc['L_star']:.6f}) ===")
    res_mlgd   = run_method(model_cond, False, NUM_X_T_LGD,    sc, N_ATTEMP_OPTIM, seed_offset=0)
    res_mlgdf  = run_method(Cos_ConsistencyModeliCT, True, NUM_X_T_LGD_CM, sc, N_ATTEMP_OPTIM, seed_offset=1000)
    all_results[name] = {"MLGD": res_mlgd, "MLGD-F": res_mlgdf}


## Results: regret against the oracle

`regret = L(x_pred) - L*` — how far each method lands from the *best achievable* loss, not from zero. On S0 this should match the main notebook's numbers (regret ≈ L2-to-x* result since `L*≈0`); on S1/S2/S3 it isolates optimization quality from target-infeasibility.

In [ ]:
def summary_row_regret(method_name, scenario_name, res, L_star):
    l2_gmm = np.array(res["l2_gmm"])
    regret = l2_gmm - L_star
    return {
        "Scenario":        scenario_name,
        "Method":          method_name,
        "L* (oracle)":     f"{L_star:.4f}",
        "L2-GMM mean":     f"{l2_gmm.mean():.4f}",
        "Regret mean":     f"{regret.mean():.4f}",
        "Regret std":      f"{regret.std():.4f}",
        "|x - x_oracle| mean": f"{np.mean(res['l2_xoracle']):.4f}",
        "|x - x_oracle| std":  f"{np.std(res['l2_xoracle']):.4f}",
        "Time mean (s)":   f"{np.mean(res['times']):.2f}",
    }

rows = []
for name, sc in scenarios.items():
    for method in ["MLGD", "MLGD-F"]:
        rows.append(summary_row_regret(method, name, all_results[name][method], sc["L_star"]))
df = pd.DataFrame(rows).set_index(["Scenario", "Method"])
display(df)


## Mode-seeking vs. mode-covering: distribution of recovered `x` across restarts

For a genuinely infeasible target (especially S2, the bimodal case) there's no reason to expect restarts to agree. Do they collapse onto a single `x` (mode-seeking, matching the oracle's preference for one component) or scatter (mode-covering / unstable)?

In [ ]:
fig, axes = plt.subplots(1, len(scenarios), figsize=(5 * len(scenarios), 4), sharey=True)
for ax, (name, sc) in zip(axes, scenarios.items()):
    x_mlgd  = [xt.float().view(-1).cpu().item() for xt in all_results[name]["MLGD"]["x_pred"]]
    x_mlgdf = [xt.float().view(-1).cpu().item() for xt in all_results[name]["MLGD-F"]["x_pred"]]
    ax.hist(x_mlgd,  bins=15, alpha=0.5, label="MLGD")
    ax.hist(x_mlgdf, bins=15, alpha=0.5, label="MLGD-F")
    ax.axvline(sc["x_oracle"], color="red", ls="--", label="x_oracle")
    ax.set_title(name); ax.set_xlabel("x_pred"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()


## MMD (optimization objective) vs. exact L2-GMM (evaluation metric): do they agree?

`optimize_LGD` is guided by a finite-sample MMD estimate; we evaluate with the exact closed-form L2-GMM distance. On an infeasible target the MMD's finite-sample bias may matter more, since there's no exact zero to converge to — check whether ranking by `final_loss` (MMD) still tracks ranking by `l2_gmm` (L2).

In [ ]:
from scipy.stats import spearmanr
for name in scenarios:
    for method in ["MLGD", "MLGD-F"]:
        res = all_results[name][method]
        mmd_vals = [fl.item() if hasattr(fl, "item") else fl for fl in res["final_loss"]]
        rho, _ = spearmanr(mmd_vals, res["l2_gmm"])
        print(f"{name:26s} | {method:7s} | Spearman(MMD, L2-GMM) = {rho:.3f}")


## Save results

In [ ]:
def to_python(val):
    if isinstance(val, torch.Tensor):
        return val.detach().cpu().tolist()
    if isinstance(val, np.ndarray):
        return val.tolist()
    if hasattr(val, "item"):
        return val.item()
    return val

out = {
    "experiment": EXPERIMENT_NAME,
    "base_experiment": BASE_EXPERIMENT_NAME,
    "seed": GLOBAL_SEED,
    "environment": env_info,
    "scenarios": {},
}
for name, sc in scenarios.items():
    out["scenarios"][name] = {
        "x_oracle": sc["x_oracle"],
        "L_star": sc["L_star"],
        "target_means": to_python(sc["mog_means"]),
        "target_variances": to_python(sc["mog_variances"]),
        "target_weights": to_python(sc["weights"]),
        "MLGD": {k: [to_python(v) for v in vs] for k, vs in all_results[name]["MLGD"].items()},
        "MLGD-F": {k: [to_python(v) for v in vs] for k, vs in all_results[name]["MLGD-F"].items()},
    }

path = os.path.join(RESULTS_DIR, f"{EXPERIMENT_NAME}_results_seed{GLOBAL_SEED}.json")
with open(path, "w") as f:
    json.dump(out, f, indent=2)
print(f"Results saved to {path}")
